# Hands-On Lab: Linear Regression From Scratch

**Week 5 Capstone Lab — ML Fundamentals & Linear Regression**

| | |
|---|---|
| **Difficulty** | ⭐⭐⭐⭐ (Advanced) |
| **Estimated Time** | 4 hours |
| **Dataset** | California Housing (sklearn) |
| **Goal** | Implement Linear Regression from scratch AND verify against sklearn |

---

## What You Will Build

| Section | Topic | Concepts |
|---|---|---|
| 0 | Setup & EDA | Data loading, exploration |
| 1 | OLS Closed-Form | Normal equations, pinv |
| 2 | Gradient Descent | Batch GD, learning rate tuning |
| 3 | Loss Surface | Contour plots, GD path |
| 4 | Polynomial Regression | Overfitting, regularization |
| 5 | Bias-Variance | Bootstrap decomposition |
| 6 | Residual Analysis | Diagnostic plots, log transform |
| 7 | sklearn Pipeline | Comparison table |
| 8 | Learning Curves | Data vs complexity tradeoff |
| 9 | Self-Check | Conceptual Q&A |

> **Ground Rule:** implement everything from scratch first, then verify with sklearn.

---
## Section 0: Setup & Overview

In [ ]:
# ── Core scientific stack ──────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import time
import warnings

# ── sklearn utilities ──────────────────────────────────────────────────────────
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score
import scipy.stats as stats

# ── Global settings ────────────────────────────────────────────────────────────
np.random.seed(42)                          # reproducibility throughout
sns.set_theme(style='whitegrid')            # consistent plot style
warnings.filterwarnings('ignore')           # suppress minor sklearn warnings
plt.rcParams['figure.dpi'] = 110           # crisp figures

print('All imports successful.')

In [ ]:
# ── Load California Housing dataset ───────────────────────────────────────────
housing = fetch_california_housing(as_frame=True)
df = housing.frame                          # full DataFrame including target

X_all = housing.data.values                 # numpy array: (20640, 8)
y_all = housing.target.values               # numpy array: (20640,)  — in $100k units
feature_names = list(housing.feature_names)

print(f'Dataset shape   : {df.shape}')
print(f'Features        : {feature_names}')
print(f'Target          : MedHouseVal (median house value, $100k)')
print(f'Target range    : [{y_all.min():.2f}, {y_all.max():.2f}]')
print()
display(df.describe().round(3))

In [ ]:
# ── Quick visual EDA — scatter matrix of selected columns ─────────────────────
# Use a random sample of 1000 rows to keep rendering fast
sample_df = df[['MedInc', 'HouseAge', 'AveRooms', 'AveOccup', 'MedHouseVal']].sample(1000, random_state=42)

axes = pd.plotting.scatter_matrix(
    sample_df, figsize=(10, 8), alpha=0.3,
    diagonal='kde', color='steelblue'
)
# Tighten up label font size
for ax in axes.flatten():
    ax.xaxis.label.set_fontsize(8)
    ax.yaxis.label.set_fontsize(8)

plt.suptitle('Scatter Matrix — California Housing (n=1000 sample)', y=1.01, fontsize=12)
plt.tight_layout()
plt.show()

# Correlation with target
print('Correlation with MedHouseVal:')
print(df.corr()['MedHouseVal'].sort_values(ascending=False).round(3))

---
## Section 1: OLS Closed-Form Solution

The **Normal Equations** give an analytic closed-form solution:

$$\hat{\theta} = (X^T X)^{-1} X^T y$$

We use the **pseudoinverse** (`np.linalg.pinv`) instead of a direct matrix inverse for numerical stability.

In [ ]:
class LinearRegressionOLS:
    """Ordinary Least Squares via Normal Equations (pure NumPy)."""

    def fit(self, X, y):
        # Prepend a column of ones to X for the bias (intercept) term
        X_b = np.column_stack([np.ones(len(X)), X])   # shape: (n, p+1)
        # Normal equations: theta = pinv(X^T X) @ X^T @ y
        # pinv handles near-singular matrices more safely than direct inv
        self.theta_ = np.linalg.pinv(X_b.T @ X_b) @ X_b.T @ y
        return self

    def predict(self, X):
        X_b = np.column_stack([np.ones(len(X)), X])   # add bias column
        return X_b @ self.theta_                        # linear combination

    @property
    def intercept_(self):
        return self.theta_[0]                           # first element = bias

    @property
    def coef_(self):
        return self.theta_[1:]                          # remaining = feature weights


# ── Scratch metric helpers ─────────────────────────────────────────────────────
def mse(y_true, y_pred):
    """Mean Squared Error from scratch."""
    return np.mean((y_true - y_pred) ** 2)

def rmse(y_true, y_pred):
    """Root Mean Squared Error from scratch."""
    return np.sqrt(mse(y_true, y_pred))

def r2(y_true, y_pred):
    """R² (coefficient of determination) from scratch.
    R² = 1 - SS_res / SS_tot
    """
    ss_res = np.sum((y_true - y_pred) ** 2)            # residual sum of squares
    ss_tot = np.sum((y_true - y_true.mean()) ** 2)     # total sum of squares
    return 1 - ss_res / ss_tot

print('LinearRegressionOLS and metric helpers defined.')

In [ ]:
# ── 80/20 train-test split ─────────────────────────────────────────────────────
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42
)

# ── Fit our OLS implementation ─────────────────────────────────────────────────
t0 = time.perf_counter()
ols = LinearRegressionOLS().fit(X_train, y_train)
ols_time = time.perf_counter() - t0

y_pred_ols = ols.predict(X_test)

# ── Fit sklearn LinearRegression for comparison ────────────────────────────────
t0 = time.perf_counter()
sk_lr = LinearRegression().fit(X_train, y_train)
sk_time = time.perf_counter() - t0

y_pred_sk = sk_lr.predict(X_test)

# ── Verify coefficients match ──────────────────────────────────────────────────
coeff_match = np.allclose(ols.coef_, sk_lr.coef_, atol=1e-6)
intercept_match = np.isclose(ols.intercept_, sk_lr.intercept_, atol=1e-6)
print(f'Coefficients match sklearn : {coeff_match}')
print(f'Intercept  matches sklearn : {intercept_match}')

# ── Report metrics ─────────────────────────────────────────────────────────────
print(f'\n--- OLS (from scratch) ---')
print(f'  MSE  : {mse(y_test, y_pred_ols):.4f}')
print(f'  RMSE : {rmse(y_test, y_pred_ols):.4f}  (in $100k units)')
print(f'  R²   : {r2(y_test, y_pred_ols):.4f}')
print(f'  Time : {ols_time*1000:.1f} ms')

In [ ]:
# ── Predicted vs Actual scatter plot ──────────────────────────────────────────
fig, ax = plt.subplots(figsize=(6, 5))

ax.scatter(y_test, y_pred_ols, alpha=0.25, s=10, color='steelblue', label='Predictions')

# Perfect-prediction diagonal: y_pred == y_true
lims = [min(y_test.min(), y_pred_ols.min()), max(y_test.max(), y_pred_ols.max())]
ax.plot(lims, lims, 'r--', linewidth=1.5, label='Perfect prediction')

ax.set_xlabel('Actual Median House Value ($100k)')
ax.set_ylabel('Predicted Median House Value ($100k)')
ax.set_title(f'OLS — Predicted vs Actual  (R²={r2(y_test, y_pred_ols):.3f})')
ax.legend()
plt.tight_layout()
plt.show()

print('Note: predictions are clipped at $500k (5.0) in the dataset — visible as horizontal stripe.')

---
## Section 2: Gradient Descent Solution

**Batch Gradient Descent** iteratively updates parameters in the direction of the negative gradient:

$$\theta \leftarrow \theta - \alpha \cdot \frac{2}{n} X^T (X\theta - y)$$

> **Important:** GD is sensitive to feature scale. We **must** normalize features before training.

In [ ]:
class LinearRegressionGD:
    """Linear Regression via Batch Gradient Descent (pure NumPy)."""

    def __init__(self, lr=0.01, n_iter=1000):
        self.lr = lr            # learning rate (step size)
        self.n_iter = n_iter    # number of gradient descent iterations

    def fit(self, X, y):
        X_b = np.column_stack([np.ones(len(X)), X])   # add bias column
        n, p = X_b.shape
        self.theta_ = np.zeros(p)                      # initialize all weights at 0
        self.loss_history_ = []                        # track MSE per iteration

        for i in range(self.n_iter):
            predictions = X_b @ self.theta_            # forward pass: X @ theta
            residuals   = predictions - y              # error vector
            gradient    = (2 / n) * X_b.T @ residuals # gradient of MSE w.r.t. theta
            self.theta_ -= self.lr * gradient          # parameter update step
            mse_val     = np.mean(residuals ** 2)      # record MSE for this iteration
            self.loss_history_.append(mse_val)

        return self

    def predict(self, X):
        X_b = np.column_stack([np.ones(len(X)), X])
        return X_b @ self.theta_

# ── Normalize features (z-score) — mandatory before GD ───────────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit on train, transform train
X_test_scaled  = scaler.transform(X_test)         # use same scaler on test (no leakage)

# ── Fit GD model ──────────────────────────────────────────────────────────────
t0 = time.perf_counter()
gd = LinearRegressionGD(lr=0.1, n_iter=1000).fit(X_train_scaled, y_train)
gd_time = time.perf_counter() - t0

y_pred_gd = gd.predict(X_test_scaled)

print(f'GD  RMSE : {rmse(y_test, y_pred_gd):.4f}')
print(f'GD  R²   : {r2(y_test, y_pred_gd):.4f}')
print(f'GD  Time : {gd_time*1000:.1f} ms')
print(f'Final loss (MSE on train): {gd.loss_history_[-1]:.4f}')

In [ ]:
# ── Plot loss curve for the good learning rate ────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(gd.loss_history_, color='steelblue', linewidth=1.5)
ax.set_xlabel('Iteration')
ax.set_ylabel('MSE (Training Loss)')
ax.set_title('Gradient Descent — Loss Curve (lr=0.1, 1000 iterations)')
ax.axhline(y=gd.loss_history_[-1], color='red', linestyle='--',
           label=f'Final MSE = {gd.loss_history_[-1]:.3f}')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── Compare 3 learning rates: too small, good, diverging ─────────────────────
learning_rates = [
    (0.001, 'Too small (0.001)', 'orange'),
    (0.1,   'Good (0.1)',        'steelblue'),
    (1.5,   'Too large (1.5)',   'red'),
]

fig, ax = plt.subplots(figsize=(8, 4))

for lr_val, label, color in learning_rates:
    model = LinearRegressionGD(lr=lr_val, n_iter=200).fit(X_train_scaled, y_train)
    # Clip loss values to avoid infinite divergence ruining the y-axis
    losses = np.clip(model.loss_history_, 0, 20)
    ax.plot(losses, label=label, color=color, linewidth=1.5)

ax.set_xlabel('Iteration')
ax.set_ylabel('MSE (clipped at 20)')
ax.set_title('Effect of Learning Rate on Gradient Descent Convergence')
ax.legend()
plt.tight_layout()
plt.show()

print('Observation:')
print('  lr=0.001 → converges, but very slowly (needs many more iterations)')
print('  lr=0.1   → converges smoothly and quickly — the sweet spot')
print('  lr=1.5   → diverges (loss explodes) — step size overshoots the minimum')

In [ ]:
# ── Compare GD final theta vs OLS solution (on normalized scale) ──────────────
# Re-fit OLS on scaled features so both operate in the same parameter space
ols_scaled = LinearRegressionOLS().fit(X_train_scaled, y_train)

print('Feature coefficients comparison (normalized feature space):')
print(f'{"Feature":<18} {"OLS theta":>12} {"GD theta":>12} {"Diff":>12}')
print('-' * 56)
names_with_bias = ['bias'] + feature_names
for name, ols_c, gd_c in zip(names_with_bias, ols_scaled.theta_, gd.theta_):
    diff = abs(ols_c - gd_c)
    print(f'{name:<18} {ols_c:>12.4f} {gd_c:>12.4f} {diff:>12.6f}')

# Check if they are close (within 1% tolerance)
converged = np.allclose(ols_scaled.theta_, gd.theta_, rtol=0.01)
print(f'\nGD converged to OLS solution (rtol=1%): {converged}')

---
## Section 3: Loss Surface Visualization

We use a **1-feature model** (`MedInc` → `MedHouseVal`) to visualize the 2D loss surface over $(\theta_0, \theta_1)$. The bowl shape confirms MSE is **convex** for linear regression.

In [ ]:
# ── Prepare 1-feature dataset (MedInc only) ───────────────────────────────────
feat_idx = feature_names.index('MedInc')           # index 0 in California housing
X1 = X_train_scaled[:, feat_idx:feat_idx+1]        # keep as 2D array, shape (n,1)
y1 = y_train

# Fit OLS to find the true optimum on this 1-feature problem
ols_1f = LinearRegressionOLS().fit(X1, y1)
theta0_opt, theta1_opt = ols_1f.theta_[0], ols_1f.theta_[1]
print(f'OLS optimum: theta0={theta0_opt:.4f}, theta1={theta1_opt:.4f}')

# ── Create grid around the optimum ───────────────────────────────────────────
t0_range = np.linspace(theta0_opt - 1.5, theta0_opt + 1.5, 80)   # intercept axis
t1_range = np.linspace(theta1_opt - 1.5, theta1_opt + 1.5, 80)   # slope axis
T0, T1  = np.meshgrid(t0_range, t1_range)                         # (80,80) grids

# ── Compute MSE at every (theta0, theta1) grid point ─────────────────────────
X1_b = np.column_stack([np.ones(len(X1)), X1])     # add bias column once
MSE_grid = np.zeros_like(T0)
for i in range(T0.shape[0]):
    for j in range(T0.shape[1]):
        theta = np.array([T0[i, j], T1[i, j]])
        preds = X1_b @ theta                        # predictions for this theta
        MSE_grid[i, j] = np.mean((preds - y1) ** 2)

print('Loss surface grid computed.')

In [ ]:
# ── Run GD on the 1-feature model and record parameter path ──────────────────
n1 = len(X1_b)
theta_path = np.zeros((1, 2))                       # start at (0, 0)
theta_curr = np.array([0.0, 0.0])
lr_path    = 0.1
n_steps    = 30                                     # record 30 steps for arrow overlay

for step in range(n_steps):
    residuals = X1_b @ theta_curr - y1
    gradient  = (2 / n1) * X1_b.T @ residuals
    theta_curr = theta_curr - lr_path * gradient
    theta_path = np.vstack([theta_path, theta_curr])

# ── Plot contour + GD path ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))

# Filled contour (color = MSE level)
cf = ax.contourf(T0, T1, MSE_grid, levels=30, cmap='RdYlGn_r', alpha=0.85)
plt.colorbar(cf, ax=ax, label='MSE')

# Contour lines for clarity
ax.contour(T0, T1, MSE_grid, levels=15, colors='white', linewidths=0.4, alpha=0.5)

# Gradient descent path as arrows
for k in range(len(theta_path) - 1):
    ax.annotate('', xy=theta_path[k+1], xytext=theta_path[k],
                arrowprops=dict(arrowstyle='->', color='white', lw=1.2))

# Starting point and OLS optimum
ax.plot(*theta_path[0], 'wo', markersize=8, label='GD start (0,0)')
ax.plot(theta0_opt, theta1_opt, 'y*', markersize=18, label='OLS optimum', zorder=5)

ax.set_xlabel('θ₀ (intercept)')
ax.set_ylabel('θ₁ (MedInc slope)')
ax.set_title('MSE Loss Surface — Bowl Shape (1-feature model)\nWhite arrows = GD path')
ax.legend(loc='upper right')
plt.tight_layout()
plt.show()

---
## Section 4: Polynomial Regression

We fit polynomials of varying degree to a noisy sine wave to observe **underfitting** (low degree) and **overfitting** (high degree).

In [ ]:
# ── Generate noisy sine-wave dataset ─────────────────────────────────────────
np.random.seed(42)
n_poly  = 25
X_poly  = np.sort(np.random.uniform(0, 1, n_poly))      # 25 random x in [0,1]
y_poly  = np.sin(2 * np.pi * X_poly) + np.random.normal(0, 0.2, n_poly)  # noisy sine

# Fine grid for plotting smooth true function and model curves
X_fine  = np.linspace(0, 1, 300)
y_true_fine = np.sin(2 * np.pi * X_fine)

# Validation set (held-out from the true function with fresh noise)
X_val   = np.sort(np.random.uniform(0, 1, 50))
y_val   = np.sin(2 * np.pi * X_val) + np.random.normal(0, 0.2, 50)

# ── Fit polynomials of each degree and record MSE ────────────────────────────
degrees     = list(range(1, 11))           # degree 1 through 10
train_mses  = []
val_mses    = []
polyfits    = {}                           # store fitted pipelines for plotting

for deg in degrees:
    pipe = Pipeline([
        ('poly',   PolynomialFeatures(deg, include_bias=False)),
        ('scaler', StandardScaler()),
        ('lr',     LinearRegression())
    ])
    pipe.fit(X_poly.reshape(-1, 1), y_poly)
    tr_mse = mse(y_poly, pipe.predict(X_poly.reshape(-1,1)))
    v_mse  = mse(y_val,  pipe.predict(X_val.reshape(-1,1)))
    train_mses.append(tr_mse)
    val_mses.append(v_mse)
    polyfits[deg] = pipe

print('Degree  Train-MSE  Val-MSE')
for d, tr, v in zip(degrees, train_mses, val_mses):
    print(f'  {d:2d}     {tr:.4f}     {v:.4f}')

In [ ]:
# ── Show polynomial fits for selected degrees ─────────────────────────────────
show_degrees = [1, 3, 5, 8, 10]
colors_poly  = ['#e41a1c', '#ff7f00', '#4daf4a', '#377eb8', '#984ea3']

fig, ax = plt.subplots(figsize=(9, 5))

# True noiseless function
ax.plot(X_fine, y_true_fine, 'k--', linewidth=2, label='True sin(2πx)')

# Training data points
ax.scatter(X_poly, y_poly, color='black', s=40, zorder=5, label='Training data')

# Each polynomial fit
for deg, color in zip(show_degrees, colors_poly):
    y_fine_pred = polyfits[deg].predict(X_fine.reshape(-1, 1))
    ax.plot(X_fine, y_fine_pred, color=color, linewidth=1.5, label=f'Degree {deg}')

ax.set_ylim(-3, 3)                         # clip extreme predictions for readability
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Polynomial Fits — Underfitting vs Overfitting')
ax.legend(loc='upper right', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# ── MSE vs Degree + Coefficient magnitudes (degree-10) ───────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Left: Train vs Validation MSE across degrees
ax1.plot(degrees, train_mses, 'o-', color='steelblue', label='Train MSE')
ax1.plot(degrees, val_mses,   's--', color='salmon',   label='Val MSE')
ax1.set_xlabel('Polynomial Degree')
ax1.set_ylabel('MSE')
ax1.set_title('Train vs Validation MSE by Degree')
ax1.set_yscale('log')                      # log scale to see differences clearly
ax1.legend()

# Highlight the sweet spot (minimum validation MSE)
best_deg = degrees[np.argmin(val_mses)]
ax1.axvline(best_deg, color='green', linestyle=':', label=f'Best deg={best_deg}')
ax1.legend()

# Right: Coefficient magnitudes for degree-10 (shows explosion in weights)
coefs_10 = polyfits[10].named_steps['lr'].coef_
coef_labels = [f'x^{i+1}' for i in range(len(coefs_10))]
ax2.bar(coef_labels, np.abs(coefs_10), color='steelblue')
ax2.set_xlabel('Feature')
ax2.set_ylabel('|Coefficient|')
ax2.set_title('Degree-10: Coefficient Magnitudes (large = overfit)')
ax2.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()
print(f'Sweet spot: degree {best_deg} (lowest validation MSE)')

In [ ]:
# ── Ridge Regularization on degree-10 model ───────────────────────────────────
# Ridge adds penalty λ||θ||² to the loss, shrinking large coefficients
alphas = [0.0, 0.1, 1.0, 10.0]            # 0.0 = no regularization
colors_ridge = ['red', 'orange', 'green', 'blue']

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(X_fine, y_true_fine, 'k--', linewidth=2, label='True sin(2πx)')
ax.scatter(X_poly, y_poly, color='black', s=40, zorder=5, label='Train data')

for alpha, color in zip(alphas, colors_ridge):
    ridge_pipe = Pipeline([
        ('poly',   PolynomialFeatures(10, include_bias=False)),
        ('scaler', StandardScaler()),
        ('ridge',  Ridge(alpha=alpha))
    ])
    ridge_pipe.fit(X_poly.reshape(-1, 1), y_poly)
    y_fine_pred = ridge_pipe.predict(X_fine.reshape(-1, 1))
    v_mse = mse(y_val, ridge_pipe.predict(X_val.reshape(-1,1)))
    label = f'α={alpha} (val MSE={v_mse:.3f})'
    ax.plot(X_fine, np.clip(y_fine_pred, -3, 3), color=color, linewidth=1.5, label=label)

ax.set_ylim(-3, 3)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.set_title('Degree-10 Polynomial: Effect of Ridge Regularization')
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

---
## Section 5: Bias-Variance Decomposition

We empirically estimate bias² and variance using **200 bootstrap datasets**:

$$\text{Bias}^2 = (\overline{\hat{f}}(x) - f(x))^2 \qquad \text{Variance} = \mathbb{E}[(\hat{f}(x) - \overline{\hat{f}}(x))^2]$$

- **High bias / low variance** → degree-1 (underfits, systematic error)
- **Low bias / high variance** → degree-9 (overfits, sensitive to data)

In [ ]:
np.random.seed(42)

# ── Setup ─────────────────────────────────────────────────────────────────────
n_boot    = 200                             # number of bootstrap datasets
n_sample  = 25                             # size of each bootstrap sample
X_test_bv = np.linspace(0, 1, 100)        # 100 fixed test points
y_true_bv = np.sin(2 * np.pi * X_test_bv) # noiseless ground truth

# Store predictions from each bootstrap model at the 100 test points
preds_deg1 = np.zeros((n_boot, 100))       # degree-1 predictions
preds_deg9 = np.zeros((n_boot, 100))       # degree-9 predictions

for b in range(n_boot):
    # Sample x uniformly; generate noisy y from the true function
    x_b = np.sort(np.random.uniform(0, 1, n_sample))
    y_b = np.sin(2 * np.pi * x_b) + np.random.normal(0, 0.3, n_sample)

    # Fit degree-1 polynomial pipeline
    p1 = Pipeline([('poly', PolynomialFeatures(1, include_bias=False)),
                   ('scaler', StandardScaler()),
                   ('lr', LinearRegression())])
    p1.fit(x_b.reshape(-1, 1), y_b)
    preds_deg1[b] = p1.predict(X_test_bv.reshape(-1, 1))

    # Fit degree-9 polynomial pipeline
    p9 = Pipeline([('poly', PolynomialFeatures(9, include_bias=False)),
                   ('scaler', StandardScaler()),
                   ('lr', LinearRegression())])
    p9.fit(x_b.reshape(-1, 1), y_b)
    preds_deg9[b] = p9.predict(X_test_bv.reshape(-1, 1))

print(f'Bootstrap complete: {n_boot} models fitted for each degree.')

In [ ]:
# ── Spaghetti plots: all 200 fitted curves per model ─────────────────────────
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), sharey=True)

for b in range(n_boot):
    ax1.plot(X_test_bv, preds_deg1[b], alpha=0.05, color='steelblue', linewidth=0.8)
    ax2.plot(X_test_bv, preds_deg9[b], alpha=0.05, color='salmon',    linewidth=0.8)

# Mean prediction across all bootstrap models
ax1.plot(X_test_bv, preds_deg1.mean(axis=0), 'b-', linewidth=2, label='Mean prediction')
ax2.plot(X_test_bv, preds_deg9.mean(axis=0), 'r-', linewidth=2, label='Mean prediction')

# True function
for ax in (ax1, ax2):
    ax.plot(X_test_bv, y_true_bv, 'k--', linewidth=2, label='True f(x)')
    ax.set_ylim(-3, 3)
    ax.set_xlabel('x')
    ax.legend(fontsize=8)

ax1.set_ylabel('y')
ax1.set_title('Degree-1 (High Bias, Low Variance)\n200 bootstrap fits')
ax2.set_title('Degree-9 (Low Bias, High Variance)\n200 bootstrap fits')
plt.suptitle('Bias-Variance Tradeoff — Spaghetti Plot', y=1.01, fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
# ── Compute bias², variance, total error empirically ─────────────────────────
def bv_decompose(preds, y_true):
    """Given preds shape (n_boot, n_test), compute bias², var, total."""
    mean_pred  = preds.mean(axis=0)                        # average prediction per test point
    bias2      = np.mean((mean_pred - y_true) ** 2)        # squared bias, averaged over test pts
    variance   = np.mean(preds.var(axis=0))                # variance of predictions, averaged
    total      = bias2 + variance                          # total error (noise excluded)
    return bias2, variance, total

b2_d1, var_d1, tot_d1 = bv_decompose(preds_deg1, y_true_bv)
b2_d9, var_d9, tot_d9 = bv_decompose(preds_deg9, y_true_bv)

print(f'{"Model":<12} {"Bias²":>8} {"Variance":>10} {"Total":>8}')
print('-' * 42)
print(f'{"Degree-1":<12} {b2_d1:>8.4f} {var_d1:>10.4f} {tot_d1:>8.4f}')
print(f'{"Degree-9":<12} {b2_d9:>8.4f} {var_d9:>10.4f} {tot_d9:>8.4f}')

# ── Bar chart ─────────────────────────────────────────────────────────────────
x_pos  = np.arange(2)
width  = 0.25
labels = ['Degree-1', 'Degree-9']

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x_pos - width, [b2_d1, b2_d9],  width, label='Bias²',    color='steelblue')
ax.bar(x_pos,         [var_d1, var_d9], width, label='Variance', color='salmon')
ax.bar(x_pos + width, [tot_d1, tot_d9], width, label='Total',    color='seagreen')
ax.set_xticks(x_pos)
ax.set_xticklabels(labels)
ax.set_ylabel('Error')
ax.set_title('Bias-Variance Decomposition\n(empirical, 200 bootstrap samples)')
ax.legend()
plt.tight_layout()
plt.show()

---
## Section 6: Residual Analysis

After fitting, we **always** inspect residuals to check if the linear model assumptions hold:
1. **Linearity** — residuals should show no pattern vs fitted values
2. **Normality** — residuals should follow a normal distribution (Q-Q plot)
3. **Homoscedasticity** — constant variance across fitted values
4. **Leverage** — identify influential data points (Cook's Distance)

In [ ]:
# ── Compute residuals from the full OLS model ──────────────────────────────────
# Use OLS fit on unscaled X (natural units) for interpretability
y_pred_train_ols = ols.predict(X_train)
residuals_raw    = y_train - y_pred_train_ols   # raw residuals
fitted_raw       = y_pred_train_ols             # fitted values

# ── Hat matrix diagonal (leverage) — h_ii = x_i^T (X^T X)^{-1} x_i ──────────
X_b_tr = np.column_stack([np.ones(len(X_train)), X_train])
XtX_inv = np.linalg.pinv(X_b_tr.T @ X_b_tr)
# Leverage for each point: h_i = x_i @ XtX_inv @ x_i
leverage = np.einsum('ij,jk,ik->i', X_b_tr, XtX_inv, X_b_tr)   # shape (n,)

# ── Cook's Distance: D_i = r_i² * h_i / (p * MSE * (1-h_i)²) ────────────────
p_params     = X_b_tr.shape[1]                  # number of parameters
mse_train    = mse(y_train, fitted_raw)
cooks_d      = (residuals_raw**2 * leverage) / (p_params * mse_train * (1 - leverage)**2 + 1e-9)

print(f'Training set size : {len(X_train)}')
print(f'Train MSE         : {mse_train:.4f}')
print(f'Max leverage      : {leverage.max():.4f}')
print(f'Max Cook\'s D      : {cooks_d.max():.4f}')

In [ ]:
# ── 4-panel diagnostic figure ─────────────────────────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# Subsample for speed (16k+ points — use 3000 random)
idx_plot = np.random.choice(len(fitted_raw), 3000, replace=False)

# --- Panel 1: Residuals vs Fitted ---
ax = axes[0, 0]
ax.scatter(fitted_raw[idx_plot], residuals_raw[idx_plot],
           alpha=0.2, s=8, color='steelblue')
ax.axhline(0, color='red', linewidth=1.5, linestyle='--')
ax.set_xlabel('Fitted Values')
ax.set_ylabel('Residuals')
ax.set_title('Residuals vs Fitted\n(check: homoscedasticity & linearity)')

# --- Panel 2: Q-Q Plot ---
ax = axes[0, 1]
res_sample = residuals_raw[idx_plot]
res_sorted = np.sort(res_sample)
theoretical_q = stats.norm.ppf(np.linspace(0.01, 0.99, len(res_sorted)))
ax.scatter(theoretical_q, res_sorted, alpha=0.3, s=6, color='steelblue')
# Reference line: perfect normality
q_lims = [theoretical_q.min(), theoretical_q.max()]
ax.plot(q_lims, [res_sorted.min() + (res_sorted.max()-res_sorted.min())*0,
                  res_sorted.max()], 'r--', linewidth=1.5)
ax.set_xlabel('Theoretical Quantiles')
ax.set_ylabel('Sample Quantiles')
ax.set_title('Q-Q Plot of Residuals\n(check: normality — deviations = fat tails)')

# --- Panel 3: Scale-Location ---
ax = axes[1, 0]
sqrt_abs_res = np.sqrt(np.abs(residuals_raw[idx_plot]))
ax.scatter(fitted_raw[idx_plot], sqrt_abs_res, alpha=0.2, s=8, color='steelblue')
ax.set_xlabel('Fitted Values')
ax.set_ylabel('√|Residuals|')
ax.set_title('Scale-Location Plot\n(horizontal band = constant variance)')

# --- Panel 4: Residuals vs Leverage ---
ax = axes[1, 1]
ax.scatter(leverage[idx_plot], residuals_raw[idx_plot],
           c=cooks_d[idx_plot], cmap='YlOrRd', alpha=0.4, s=8)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('Leverage (h_ii)')
ax.set_ylabel('Residuals')
ax.set_title('Residuals vs Leverage\n(color = Cook\'s Distance)')

plt.suptitle('Linear Regression Diagnostic Plots — Original Target', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

print('Interpretation:')
print('  Panel 1: Fan-shaped pattern → heteroscedasticity (variance increases with fitted value)')
print('  Panel 2: Heavy right tail → residuals are right-skewed, not perfectly normal')
print('  Panel 3: Increasing trend → confirms non-constant variance')
print('  Panel 4: A few high-leverage points may be pulling the fit')

In [ ]:
# ── Fix: log-transform the target, refit, redo diagnostics ───────────────────
y_train_log = np.log1p(y_train)            # log(1 + y) — safe for y near 0
y_test_log  = np.log1p(y_test)

ols_log = LinearRegressionOLS().fit(X_train, y_train_log)
y_pred_log_train = ols_log.predict(X_train)
residuals_log    = y_train_log - y_pred_log_train
fitted_log       = y_pred_log_train

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

# Residuals vs Fitted after log transform
ax1.scatter(fitted_log[idx_plot], residuals_log[idx_plot],
            alpha=0.2, s=8, color='seagreen')
ax1.axhline(0, color='red', linewidth=1.5, linestyle='--')
ax1.set_xlabel('Fitted Values (log scale)')
ax1.set_ylabel('Residuals')
ax1.set_title('Residuals vs Fitted — log(y)\n(more homoscedastic?)')

# Q-Q after log transform
res_log_sorted = np.sort(residuals_log[idx_plot])
ax2.scatter(theoretical_q[:len(res_log_sorted)], res_log_sorted,
            alpha=0.3, s=6, color='seagreen')
ax2.plot([theoretical_q.min(), theoretical_q.max()],
         [res_log_sorted.min(), res_log_sorted.max()],
         'r--', linewidth=1.5)
ax2.set_xlabel('Theoretical Quantiles')
ax2.set_ylabel('Sample Quantiles')
ax2.set_title('Q-Q Plot — log(y) Residuals\n(closer to normal?)')

plt.suptitle('After Log-Transform of Target', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

# Compare R² on test set
y_pred_log_test = ols_log.predict(X_test)
print(f'R² before log transform : {r2(y_test, ols.predict(X_test)):.4f}')
print(f'R² after  log transform : {r2(y_test_log, y_pred_log_test):.4f}')
print('Note: R² values are on different scales (raw vs log), so direct comparison is informational only.')

---
## Section 7: sklearn Comparison and Pipeline

A `Pipeline` chains preprocessing and modeling steps, ensuring no data leakage. Here we compare all four implementations head-to-head.

In [ ]:
# ── sklearn Pipeline: StandardScaler → LinearRegression ───────────────────────
sk_pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('lr',     LinearRegression())
])

t0 = time.perf_counter()
sk_pipe.fit(X_train, y_train)
pipe_time = time.perf_counter() - t0

y_pred_pipe = sk_pipe.predict(X_test)

print('sklearn Pipeline fitted.')
print(f'Pipeline R²   : {r2_score(y_test, y_pred_pipe):.4f}')
print(f'Pipeline RMSE : {mean_squared_error(y_test, y_pred_pipe)**0.5:.4f}')

In [ ]:
# ── Comprehensive comparison of all 4 implementations ─────────────────────────
results = {
    'OLS (scratch)':    (y_pred_ols,    ols_time),
    'GD (scratch)':     (y_pred_gd,     gd_time),
    'sklearn LR':       (y_pred_sk,     sk_time),
    'sklearn Pipeline': (y_pred_pipe,   pipe_time),
}

print(f'{"Method":<22} {"R²":>8} {"RMSE":>8} {"Time (ms)":>12}')
print('-' * 55)

for method, (preds, elapsed) in results.items():
    r2_val   = r2(y_test, preds)
    rmse_val = rmse(y_test, preds)
    print(f'{method:<22} {r2_val:>8.4f} {rmse_val:>8.4f} {elapsed*1000:>12.1f}')

print('\nAll R² and RMSE values should be essentially identical.')
print('(GD may differ slightly if not fully converged.)')

In [ ]:
# ── Visual comparison bar chart ────────────────────────────────────────────────
methods  = list(results.keys())
r2_vals  = [r2(y_test, p) for p, _ in results.values()]
rmse_vals= [rmse(y_test, p) for p, _ in results.values()]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

bars1 = ax1.bar(methods, r2_vals, color=['steelblue','salmon','seagreen','gold'])
ax1.set_ylabel('R²')
ax1.set_title('R² — All Implementations')
ax1.set_ylim(0, 1)
ax1.tick_params(axis='x', rotation=20)
for bar, val in zip(bars1, r2_vals):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
             f'{val:.4f}', ha='center', va='bottom', fontsize=9)

bars2 = ax2.bar(methods, rmse_vals, color=['steelblue','salmon','seagreen','gold'])
ax2.set_ylabel('RMSE ($100k)')
ax2.set_title('RMSE — All Implementations')
ax2.tick_params(axis='x', rotation=20)
for bar, val in zip(bars2, rmse_vals):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
             f'{val:.4f}', ha='center', va='bottom', fontsize=9)

plt.suptitle('All Implementations Compared — California Housing Test Set', fontsize=12)
plt.tight_layout()
plt.show()

---
## Section 8: Learning Curves

Learning curves reveal whether a model is **limited by complexity (underfitting)** or **limited by data (needs more samples)**.

- Train error ≈ Val error at a **high value** → underfitting
- Large gap between train and val error → overfitting

In [ ]:
# ── Compute learning curves for degree-1, 3, 9 polynomials ───────────────────
# Use a 2000-sample subset of California housing for speed
n_lc      = 2000
X_lc      = X_all[:n_lc]
y_lc      = y_all[:n_lc]
train_sizes = np.linspace(50, 500, 10, dtype=int)   # 50 to 500 samples

lc_degrees = [1, 3, 9]
lc_colors  = ['steelblue', 'seagreen', 'salmon']
lc_results = {}

for deg in lc_degrees:
    pipe_lc = Pipeline([
        ('poly',   PolynomialFeatures(deg, include_bias=False)),
        ('scaler', StandardScaler()),
        ('lr',     LinearRegression())
    ])
    # learning_curve returns (train_sizes, train_scores, val_scores)
    # scoring='neg_mean_squared_error' → negate to get MSE
    sizes, tr_sc, val_sc = learning_curve(
        pipe_lc, X_lc, y_lc,
        train_sizes=train_sizes,
        cv=5,
        scoring='neg_mean_squared_error',
        n_jobs=-1
    )
    lc_results[deg] = {
        'sizes':   sizes,
        'train':   -tr_sc.mean(axis=1),   # negate: back to positive MSE
        'val':     -val_sc.mean(axis=1)
    }

print('Learning curves computed for degrees:', lc_degrees)

In [ ]:
# ── Plot learning curves for all 3 degrees ────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=False)

for ax, deg, color in zip(axes, lc_degrees, lc_colors):
    lc = lc_results[deg]
    ax.plot(lc['sizes'], lc['train'], 'o-', color=color,    label='Train MSE')
    ax.plot(lc['sizes'], lc['val'],   's--', color='black', label='Val MSE (CV)')
    ax.set_xlabel('Training Set Size')
    ax.set_ylabel('MSE')
    ax.set_title(f'Degree-{deg} Polynomial\nLearning Curve')
    ax.legend(fontsize=8)

plt.suptitle('Learning Curves — Degree 1, 3, 9 (California Housing subset)', fontsize=12, y=1.01)
plt.tight_layout()
plt.show()

print('Interpretation:')
print('  Degree-1 : Both curves plateau at a high error → underfitting. More data will NOT help.')
print('  Degree-3 : Gap narrows as data grows → more data helps. Model has right complexity.')
print('  Degree-9 : Large gap (overfit). More data reduces gap — complexity is too high for 50 samples.')

---
## Section 9: Self-Check Questions

Click the triangles to reveal the answers only after attempting each question.

---

**Q1.** Your OLS and GD implementations give slightly different coefficients after 1000 iterations. What does this indicate?

<details>
<summary>Answer</summary>

GD has **not yet converged** to the optimum. The OLS normal equations give the exact closed-form solution in one shot. GD approaches it iteratively — 1000 iterations may be insufficient, especially with a small learning rate. Solutions: increase `n_iter`, raise `lr` (carefully), or use a convergence tolerance check (`|loss[t] - loss[t-1]| < 1e-6`) to stop early. Both should match OLS exactly given enough iterations and an appropriate learning rate.
</details>

---

**Q2.** The MSE loss surface for linear regression is a perfect bowl (strictly convex). Why does this guarantee GD finds the global minimum?

<details>
<summary>Answer</summary>

A **convex function** has exactly one minimum — the global minimum. Any local descent step moves toward this unique point. For linear regression, MSE = ||Xθ - y||² / n is a quadratic function in θ, which is always convex (the Hessian = 2X^TX/n is positive semi-definite). This means GD **cannot get stuck** in a local minimum or a saddle point — every gradient step reduces loss monotonically (for sufficiently small lr), guaranteeing convergence to the global optimum. This is **not** true for neural networks, which have non-convex loss surfaces.
</details>

---

**Q3.** Degree-9 has lower training error but higher validation error than degree-3. Explain using bias-variance decomposition.

<details>
<summary>Answer</summary>

**Degree-3:** moderate complexity → moderate bias (can't fit the exact curve) + low variance (stable across different training sets). Net result: reasonable validation error.

**Degree-9:** very flexible → near-zero training error (bias≈0, it memorizes the training points) but extremely high variance — the model changes dramatically with small changes in training data. On new validation data, these wild swings produce large errors.

Total expected error = Bias² + Variance + irreducible noise. Degree-9 wins on bias but loses badly on variance, giving a higher total validation error. This is the **bias-variance tradeoff** — the optimal degree minimizes the sum, not just one term.
</details>

---

**Q4.** Your Q-Q plot shows a heavy right tail in the residuals. What does this mean, and would you fix it?

<details>
<summary>Answer</summary>

A **heavy right tail** means the residuals have more extreme positive values than a normal distribution predicts — i.e., the model systematically under-predicts some high-value houses, creating large positive residuals. This violates the normality assumption of OLS.

**When to fix:** for inference (hypothesis tests, confidence intervals on coefficients), normality matters — fix it. For pure prediction, mild non-normality is often acceptable since OLS is still BLUE (Best Linear Unbiased Estimator) by the Gauss-Markov theorem even without normality.

**How to fix:** log-transform the target (`log(y)`) compresses the right tail. Alternatively, use robust regression (Huber loss) or a model that doesn't assume normality (GBM, Random Forest).
</details>

---

**Q5.** Learning curves show both training error and validation error plateau at a high value (large gap is absent). Is this overfitting or underfitting? What should you do?

<details>
<summary>Answer</summary>

This is **underfitting (high bias)**. The signature is: train ≈ val (small gap), but both are **too high**. The model lacks the capacity to capture the true relationship — more data will NOT help because the bottleneck is model complexity, not data volume.

**What to do:**
1. **Increase model complexity** — use polynomial features, add interaction terms, or switch to a more powerful model (GBM, neural network)
2. **Add informative features** — feature engineering may expose signal the linear model can use
3. **Remove regularization** — if you added Ridge/Lasso, reduce alpha
4. **Check for data quality issues** — mislabeled targets or irrelevant features suppress the signal
</details>